# Project 4 — Concept Bottleneck Models for skin lesion diagnosis (derm7pt)
Responsible AI 2026. Assignment: `docs/assignment4_part1.pdf`.

**Pipeline:** image $x$ → 7-pt criteria concepts $\hat c = g(x)$ → diagnosis $\hat y = f(\hat c)$ (5 classes: BCC, NEV, MEL, MISC, SK).

In [ ]:
from pathlib import Path
import random

import numpy as np
import pandas as pd
import torch

## Config

In [ ]:
CONFIG = {
    "seed": 42,
    # data
    "data_root": Path("data/derm7pt"),
    "image_type": "derm",          # "derm" (dermoscopic) | "clinic" (clinical photograph)
    "image_size": 224,
    # model
    "backbone": "resnet50",
    "pretrained": True,
    "label_hidden": 0,             # 0 = linear c -> y, >0 = MLP hidden size
    # training
    "regime": "joint",             # "independent" | "sequential" | "joint"
    "batch_size": 32,
    "epochs": 50,
    "lr": 1e-4,
    "weight_decay": 1e-4,
    "concept_loss_weight": 1.0,    # lambda in joint training: L_y + lambda * L_c
    "patience": 10,
    "output_dir": Path("results"),
}

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

set_seed(CONFIG["seed"])
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
device

## 1. Data
Diagnosis grouping and concept categories follow Kawahara et al. (2019) / the official
[derm7pt repo](https://github.com/jeremykawahara/derm7pt). We use the **official splits**
(413 train / 203 val / 395 test); val for model selection only, test once per final model.

In [ ]:
# --- Diagnosis -> 5 classes -------------------------------------------------
DIAGNOSIS_CLASSES = ["BCC", "NEV", "MEL", "MISC", "SK"]

DIAGNOSIS_MAP = {
    "basal cell carcinoma": "BCC",
    "blue nevus": "NEV",
    "clark nevus": "NEV",
    "combined nevus": "NEV",
    "congenital nevus": "NEV",
    "dermal nevus": "NEV",
    "recurrent nevus": "NEV",
    "reed or spitz nevus": "NEV",
    "melanoma": "MEL",
    "melanoma (in situ)": "MEL",
    "melanoma (less than 0.76 mm)": "MEL",
    "melanoma (0.76 to 1.5 mm)": "MEL",
    "melanoma (more than 1.5 mm)": "MEL",
    "melanoma metastasis": "MEL",
    "dermatofibroma": "MISC",
    "lentigo": "MISC",
    "melanosis": "MISC",
    "miscellaneous": "MISC",
    "vascular lesion": "MISC",
    "seborrheic keratosis": "SK",
}

# --- 7-point checklist criteria -> categorical concepts ---------------------
# Each concept maps raw meta.csv values to a small set of categories.
CONCEPTS: dict[str, dict[str, str]] = {
    "pigment_network": {
        "absent": "absent", "typical": "typical", "atypical": "atypical",
    },
    "streaks": {
        "absent": "absent", "regular": "regular", "irregular": "irregular",
    },
    "pigmentation": {
        "absent": "absent",
        "diffuse regular": "regular", "localized regular": "regular",
        "diffuse irregular": "irregular", "localized irregular": "irregular",
    },
    "regression_structures": {
        "absent": "absent",
        "blue areas": "present", "white areas": "present",
        "combinations": "present",
    },
    "dots_and_globules": {
        "absent": "absent", "regular": "regular", "irregular": "irregular",
    },
    "blue_whitish_veil": {
        "absent": "absent", "present": "present",
    },
    "vascular_structures": {
        "absent": "absent",
        "arborizing": "regular", "comma": "regular", "hairpin": "regular",
        "within regression": "regular", "wreath": "regular",
        "dotted": "irregular", "linear irregular": "irregular",
    },
}

# Ordered category list per concept (absent first).
CONCEPT_CATEGORIES: dict[str, list[str]] = {
    name: list(dict.fromkeys(mapping.values())) for name, mapping in CONCEPTS.items()
}


def load_metadata(root: str | Path) -> pd.DataFrame:
    """Load meta.csv and add integer-encoded `label` and `<concept>_idx` columns."""
    root = Path(root)
    meta = pd.read_csv(root / "meta" / "meta.csv")

    meta["label"] = meta["diagnosis"].map(DIAGNOSIS_MAP).map(DIAGNOSIS_CLASSES.index)
    for name, mapping in CONCEPTS.items():
        cats = CONCEPT_CATEGORIES[name]
        meta[f"{name}_idx"] = meta[name].map(mapping).map(cats.index)

    unmapped = meta[["label"] + [f"{c}_idx" for c in CONCEPTS]].isna().any(axis=1)
    if unmapped.any():
        raise ValueError(f"{unmapped.sum()} rows have unmapped diagnosis/concept values")
    return meta


def load_splits(root: str | Path) -> dict[str, pd.Index]:
    """Official derm7pt split indexes (row indices into meta.csv)."""
    meta_dir = Path(root) / "meta"
    return {
        split: pd.read_csv(meta_dir / f"{name}_indexes.csv")["indexes"].values
        for split, name in [("train", "train"), ("val", "valid"), ("test", "test")]
    }

In [ ]:
meta = load_metadata(CONFIG["data_root"])
splits = load_splits(CONFIG["data_root"])
print({k: len(v) for k, v in splits.items()})
meta.loc[splits["train"], "label"].map(dict(enumerate(DIAGNOSIS_CLASSES))).value_counts()

## 2. Model (concept model $g$, label model $f$)

In [ ]:
# TODO

## 3. Training: independent / sequential / joint

| Regime | $g: x \to c$ | $f: c \to y$ |
|---|---|---|
| Independent | trained on $(x, c)$ | trained on ground-truth $c$; at test time fed $\hat c$ |
| Sequential | trained on $(x, c)$ | trained on $\hat c = g(x)$, $g$ frozen |
| Joint | trained together with $L_y + \lambda L_c$ | |

In [ ]:
# TODO

## 4. Standard performance assessment

In [ ]:
# TODO

## 5. Exploring concept explanations

In [ ]:
# TODO

## 6. Concept interventions

In [ ]:
# TODO

## 7. Use cases & success criteria
See `docs/use_cases.md`.